# Bedside

> Processing utilities for bedside/ICU data

In [ ]:
#| default_exp bedside

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import yaml, lxml.etree as ET,  numpy as np, zarr, dask.array as da, sys, datetime as dt, warnings, pandas as pd, torch

from collections import Counter
from tqdm.auto import tqdm
from pathlib import Path
from copy import deepcopy

from scipy.signal import resample
import torchaudio.functional as F_audio


import datetime as dt
from typing import Union
from collections import Counter

from torch.utils.data import Dataset
from wavefm.data_preprocessing import calculate_samples_mp, interpolate_nan_clip, calculate_samples_forecast_mp, calculate_samples_forecast_df
from wavefm.signal import butterworth, iqr_normalization, resample_waveform, create_patch_b2b, ecg_clean_beats
from scipy.ndimage import median_filter

/opt/anaconda3/envs/timeflies/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#| export
## constants

ALL_FREQUENCY_FILTERS = {
            #'ABP':[0.05, 20], # lowpass
            'II':[None,0.3], # bandpass
            'V':[None,0.3], # bandpass
            'I':[None,0.3], # bandpass
            'III':[None,0.3], # bandpass
            'AVR':[None,0.3], # bandpass
            'AVF':[None,0.3], # bandpass
            'PLETH':[0.1, 15], # lowpass
            'RESP':[0.1, 15] # lowpass
            }

CLIP_INTERPOLATE_RANGES = {'ABP': {"phys_range":None, "percentiles":None}, 
                           'ART1': {"phys_range":None, "percentiles":None}, 
                           'II': {"phys_range":None, "percentiles":None}, 
                           'V': {"phys_range":None, "percentiles":None}, 
                           'I': {"phys_range":None, "percentiles":None}, 
                           'III': {"phys_range":None, "percentiles":None},
                           'AVR': {"phys_range":None, "percentiles":None},
                           'AVF': {"phys_range":None, "percentiles":None},
                           'PLETH': {"phys_range":None, "percentiles":None},
                           'RESP': {"phys_range":None, "percentiles":None},
                           'SPO2': {"phys_range":None, "percentiles":None},
                           }

BEDMASTER_CALIBRATIONS = {'ART1':0.2,
                          'ART2':0.2,
                          'I':2.44,
                          'II':2.44,
                          'III':2.44,
                          'V':2.44
                          }

In [ ]:
#| export
def count_waveforms(file_header_dir:list # Directory path of the yaml header files to be counted.
                    )->Counter: # Counter dictionary of the waveforms
    """Function to count the waveforms in the yaml header files"""
    c = Counter()
    for p in tqdm(file_header_dir):
        with open(p, 'r') as f:
            file = yaml.safe_load(f)
        if file is not None:
            if bool(file.get('WaveformData')):
                for key in file['WaveformData']:
                    c[key] += 1
    return c

In [ ]:
#| export
def count_units(file_headers:list # Directory path of the yaml header files to be counted.
                    )->Counter: # Counter dictionary of the units
    """Function to count the units/icus in the yaml header files"""
    c = Counter()
    for p in tqdm(file_headers):
        with open(p, 'r') as f:
            file = yaml.safe_load(f)
        if file is not None:
            if bool(file.get('Unit')):
                c[file['Unit']] += 1
    return c

In [ ]:
#| export
NULL_VALUE = -32768 # int16 smallest number
RESAMPLE_WAVES = {'I':(120,240), 'II':(120,240), 'III':(120,240), 'V':(120,240)}
ALL_CHANNELS = ['I',
 'II',
 'III',
 'V',
 'RR',
 'SPO2',
 'CO2',
# '',
 'ART1',
 'AVR',
 'AVL',
 'AVF',
 'V6',
 'O2',
 'V2',
 'CVP2',
 'ICP2',
 'ART2',
 'PA2',
 'ICP1',
 'CVP1',
 'V5',
 'CVP3',
 'PA1',
 'SP4',
 'CVP4',
 'PA4',
 'SP2',
 'PA3',
 'LA4',
 'RA2',
 'UVC2',
 'UAC1',
 'ART3',
 'ART4',
 'UAC4',
 'FEM1',
 'ICP4',
 'ICP3',
 'FEM2',
 'SP1',
 'VNT_PRES',
 'VNT_FLOW',
 'V3',
 'LA1',
 'LA2',
 'SP3',
 'N2OXP',
 'RM_PRES',
 'RM_FLOW',
 'V4',
 'UAC2',
 'LA6',
 'RA1',
 'RA4',
 '251',
 'FEM3',
 'ICG',
 'FEM4']

def parse_to_zarr(file_path:str, # path of xml file to parse
                  file_header_directory:str, # path to file header directory
                  write_data_dir:str, # directory to write zarr files to
                  null_value:int=NULL_VALUE, # null value to fill in segment discontinuities
                  resample_waves:dict = RESAMPLE_WAVES, # dictionary of waveform keys:tuple to resample the waveforms to
                  overwrite:bool=True, # boolean indicator to overwrite existing zarr files
                  waveforms:list=['I','II','III','V','RR','SPO2'], # list of waveforms to parse from xml
                  verbose:bool = False # indicator to output verbose and use tqdm
                  ):
    # read in header to get relevant metadata
    header_path = str(file_header_directory/Path(file_path).stem)+'.yaml'
    with open(header_path, 'r') as f:
        header = yaml.safe_load(f)
    # get start/end times from file to create dataset
    ts_context = ET.iterparse(file_path, events=("start","end",), remove_blank_text=True, huge_tree=True, recover=True)
    ts_context = iter(ts_context)
    _, ts_root = next(ts_context)
    # iterate through file to get start and end times
    prev_start = False
    end_time = 0
    last_valid_elem = False
    if verbose:
        print("Finding start and end time for file")
    for event, elem in tqdm(ts_context,disable=(not verbose)):
        if elem.tag == 'Waveforms' and elem.find('WaveformData') is not None and event=='end': # ensure that the tag has waveform data in it! 
            if not prev_start:
                start_time = int(elem.attrib['CollectionTimeUTC'])
                prev_start=True
            end_time = max(int(elem.attrib['CollectionTimeUTC']), end_time)
            last_valid_elem = deepcopy(elem)
        if event=='end':
            ts_root.clear() # free up memory
        else:
            continue
    del ts_context
    
    n_seconds_to_add = 0
    prev_child = 0
    if not prev_start:
        print(f'File {file_path} has no waveform data.')
        return 0

    #elem is the last element from previous loop, iterate through its waveformdata children
    for child in last_valid_elem.iterchildren():
        if prev_child == 0:
            # do nothing
            if child.text is not None:
                n_seconds_to_add = int(child.attrib['Samples']) / int(child.attrib['SampleRate'])
                prev_child = n_seconds_to_add
        else:
            # assert that the prev_child n seconds to add is the same as the other children
            if child.text is not None:
                assert prev_child == int(child.attrib['Samples']) / int(child.attrib['SampleRate']), "The last elements children do not have the same time interval"

    # add the final number of seconds to the end_time
    if verbose:
        print("Seconds to add: ", n_seconds_to_add)
    end_time = int(end_time + n_seconds_to_add)
    if verbose:
        print(f"Start time: {start_time}, end time: {end_time}")

    store = zarr.DirectoryStore(str(write_data_dir/Path(file_path).stem) + '.zarr')
    if overwrite:
        root_grp = zarr.group(store, overwrite=True)
    else:
        # this should not replace data, so we can continue to append to this group.
        root_grp = zarr.group(store, overwrite=False)
    # get maximum waveform frequencies from header file
    waveform_frequencies = {}
    waveform_chunks = {}
    for k,v in header['WaveformData'].items():
        if k in waveforms:
            waveform_frequencies[k] = max(v['SampleRate']) if isinstance(v['SampleRate'], list) else v['SampleRate']
            samples_per_segment = max(v['SamplesPerSegment']) if isinstance(v['SamplesPerSegment'], list) else v['SamplesPerSegment']
            #waveform_chunks[k] = guess_chunks((v['SampleSegments'] * samples_per_segment,), typesize=2)
            # create groups in dataset
            waveform_length = (end_time-start_time) * waveform_frequencies[k]
            a = root_grp.create_dataset(f"{k}_pre_chunk", shape=(waveform_length,), fill_value=null_value, chunks='auto', dtype=np.int16)
            a.attrs['sample_time_start'] = start_time
            a.attrs['sample_rate'] = waveform_frequencies[k]
            a.attrs['was_resampled_within_file'] = True if isinstance(v['SampleRate'], list) else False
    if verbose:
        print(f"Waveform frequencies: {waveform_frequencies}")
        print(f"Guessed chunk sizes: {waveform_chunks}")
    all_records = {k:[] for k in header['WaveformData'].keys() if k in waveforms}
    idx_tracker = {k:{'start_idx':0, 'end_idx':0} for k in header['WaveformData'].keys() if k in waveforms}
    prev_end_time_tracker = {k:start_time for k in header['WaveformData'].keys() if k in waveforms}

    context = ET.iterparse(file_path, events=("start","end"), remove_blank_text=True, huge_tree=True, recover=True)
    context = iter(context)
    _, main_root = next(context)
    idx = 1

    file_dup_counter = 0

    for event, elem in tqdm(context, disable=(not verbose)):
        if elem.text is not None and bool(elem.attrib.get('Label')) and elem.attrib['Label'] in waveforms and elem.tag == 'WaveformData' and event=='end':
            sample_start = int(elem.getparent().attrib['CollectionTimeUTC'])
            sample_data = elem.text.strip(',')
            n_samples = sample_data.count(',') + 1
            supposed_length = int(elem.attrib['Samples']) # reported number of samples
            wave_frequency = int(elem.attrib['SampleRate'])
            if n_samples != supposed_length:
                sample_data = (sample_data + (f",{null_value}" * (supposed_length - n_samples)))
                n_samples = sample_data.count(',') + 1
            sample_data = map(int, sample_data.split(','))
            if wave_frequency != waveform_frequencies[elem.attrib['Label']]:
                # resample/upsample
                sample_data = list(sample_data)
                sample_data = resample(sample_data, int((supposed_length/wave_frequency)*waveform_frequencies[elem.attrib['Label']]))
                n_samples = len(sample_data)
            if prev_end_time_tracker[elem.attrib['Label']] <= sample_start:
                if prev_end_time_tracker[elem.attrib['Label']] < sample_start:
                    # prepend data
                    sample_data = list(sample_data)
                    missing_start_vals = int((sample_start - prev_end_time_tracker[elem.attrib['Label']]) * waveform_frequencies[elem.attrib['Label']])
                    sample_data = [null_value] * missing_start_vals + sample_data
                    n_samples = len(sample_data)
                all_records[elem.attrib['Label']].extend(sample_data)
                idx_tracker[elem.attrib['Label']]['end_idx'] = idx_tracker[elem.attrib['Label']]['end_idx'] + n_samples
                prev_end_time_tracker[elem.attrib['Label']] = int(sample_start + supposed_length/wave_frequency)
                idx += 1
                if idx % 1_000_000 == 0:
                    if verbose:
                        print(f"Writing data at idx: {idx}")
                    for waveform, data in all_records.items():
                        a = root_grp[f"{waveform}_pre_chunk"]
                        a[idx_tracker[waveform]['start_idx']:idx_tracker[waveform]['end_idx']] = np.array(data, dtype=np.int16)
                        idx_tracker[waveform]['start_idx'] = idx_tracker[waveform]['end_idx']
                    del all_records
                    all_records = {k:[] for k in header['WaveformData'].keys() if k in waveforms}
            else:
                file_dup_counter += 1
        # clean up https://stackoverflow.com/questions/12160418/why-is-lxml-etree-iterparse-eating-up-all-my-memory/12161078#12161078
        if event=='end':
            elem.clear()
            main_root.clear()
        # for ancestor in elem.xpath('ancestor-or-self::*'):
        #     while ancestor.getprevious() is not None:
        #         try:
        #             del ancestor.getparent()[0]
        #         except:
        #             pass
        # write every so often
        # if idx % 1_000_000 == 0:
        #     if verbose:
        #         print(f"Writing data at idx: {idx}")
        #     for waveform, data in all_records.items():
        #         a = root_grp[f"{waveform}_pre_chunk"]
        #         a[idx_tracker[waveform]['start_idx']:idx_tracker[waveform]['end_idx']] = np.array(data, dtype=np.int16)
        #         idx_tracker[waveform]['start_idx'] = idx_tracker[waveform]['end_idx']
        #     del all_records
        #     all_records = {k:[] for k in header['WaveformData'].keys() if k in waveforms}
        # # clean up
        # if elem.getnext() is None:
        #     elem.clear()
        #     if hasattr(elem, "getprevious"):
        #         while (elem.getprevious() is not None and elem.getparent() is not None):
        #             del elem.getparent()[0]

    del context
    #write one last time
    for waveform, data in all_records.items():
        if prev_end_time_tracker[waveform] < end_time:
            # append at end if missing
            missing_end_vals = int((end_time - prev_end_time_tracker[waveform]) * waveform_frequencies[waveform])
            data = data + [null_value] * missing_end_vals
            idx_tracker[waveform]['end_idx'] = idx_tracker[waveform]['end_idx'] + missing_end_vals
        a = root_grp[f"{waveform}_pre_chunk"]
        a[idx_tracker[waveform]['start_idx']:idx_tracker[waveform]['end_idx']] = np.array(data, dtype=np.int16)
    del all_records
    #_ = gc.collect()
    if verbose:
        print(f"***Rechunking data***")
    # rechunk
    for waveform in list(root_grp.array_keys()):
        a = root_grp[waveform]
        da_array = da.from_zarr(a)
        da_array = da_array.rechunk(chunks='auto')
        da_array.to_zarr(url=store, component=waveform.split('_pre_chunk')[0], compute=True)
        for k,v in root_grp[waveform].attrs.asdict().items():
            root_grp[waveform.split('_pre_chunk')[0]].attrs[k] = v
        del root_grp[waveform]
    if file_dup_counter != 0:
        root_grp.attrs['duplicate_count'] = file_dup_counter
        print(f"The file: {file_path} has {file_dup_counter} duplicate entries. The first sample was taken if a duplicate was detected.")
    zarr.consolidate_metadata(store)
    return root_grp

def error_callback_handler(error # An error raised from an exception 
                           ):
    """
    Function to handle/print errors in multiproccessing async methods. =
    """
    print(f'Error: {error}', flush=True)

def chunks(lst:list, # list to chunk
           n # how many chunks to create
           )->list: # chunked list
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

In [ ]:
#| export
def clean_dict(d:dict # Dictionary of keys and values to unlist values in lists from
               )->dict: # Dictionary with values that were length 1 lists as single elements
    """
    Recursive function to unlist length 1 lists in dictionary values 
    """
    if isinstance(d, dict):
        return {k:(v[0] if isinstance(v, list) and len(v) == 1 else clean_dict(v)) for k,v in d.items()}
    else:
        return d

def unix_to_iso(d:Union[int,str] # Unix timestamp to convert to isoformat
                )->str: # String of isoformat datetime representation
    """
    Function to convert unix timestamps to isoformat 
    """
    if d == sys.maxsize or d == 0:
        return ''
    else:
        return dt.datetime.fromtimestamp(d).isoformat()

HEADER_TEMPLATE_DICT = {'Filename':'',
                 'Size':'',
                 'Unit':'',
                 'Bed':'',
                 'FamilyType':'',
                 'PatientNames':[],
                 'PatientIDs':[],
                 'TotalSegments':0, 
                 'WaveformStartTime':sys.maxsize,
                 'WaveformEndTime':0,
                 'VitalStartTime':sys.maxsize,
                 'VitalEndTime':0,
                 'WaveformData':{},
                 'VitalSign':{},
                 'TotalAlarms':0
                }

def parse_bedmaster_header(file_path:str, # File path of bedmaster xml file
                           d:dict=HEADER_TEMPLATE_DICT #  Template dictionary to input xml file details into
                           )->dict: # Dictionary of the file's metadata.
    """
    Function to iteratively parse a bedmaster xml file. This function will try to fix broken xml files via the
    recover=True parameter in iterparse. It is specific to bedmaster data files in the bedmaster directory at
    /ar/scion/projects/mscic1/data/bedmaster/bedmaster_decompressed/. This attempts to get file, patient, waveform,
    vital, and alarm metadata from a single xml file. Additional metadata could be extracted by adding additional
    cases to this function. 
    """
    d = deepcopy(d)
    for event, elem in ET.iterparse(file_path, events=("start","end"), remove_blank_text=True, huge_tree=True, recover=True):
        if event == 'start':
            if elem.tag in set(['Filename', 'Size', 'Unit', 'Bed', 'FamilyType']):
                # unique text values that should not change
                assert not bool(d.get(elem.tag)) or d.get(elem.tag) == elem.text or not bool(elem.text), f"Only one {elem.tag} should be in the file. Detected {elem.text} and {d.get(elem.tag)}"
                d[elem.tag] = elem.text if elem.tag != 'Size' else int(elem.text)
            if elem.tag in ['PatientName', 'PatientUpdate']:
                name = elem.text if elem.tag == 'PatientName' else elem.attrib.get('NewPatientName')
                id_ = elem.attrib['ID'] if elem.tag == 'PatientName' else elem.attrib.get('NewPatientID')
                if id_:
                    d['PatientIDs'] = list(set(d['PatientIDs']) | {str(id_)})
                if name:
                    d['PatientNames'] = list(set(d['PatientNames']) | {elem.text})
            if elem.tag == 'Segment':
                # count segments
                d['TotalSegments'] = max(d['TotalSegments'], int(elem.attrib['N']) + 1)
            if (elem.tag == 'Waveforms' and elem.find('WaveformData') is not None) or (elem.tag == 'VitalSigns' and elem.find('VitalSign') is not None):
                # find vital/waveforms with children/waveform data to get start time and end time
                if elem.tag == 'Waveforms':
                    d['WaveformStartTime'] = min(d['WaveformStartTime'], int(elem.attrib['CollectionTimeUTC']))
                    d['WaveformEndTime'] = max(d['WaveformEndTime'], int(elem.attrib['CollectionTimeUTC']))
                else:
                    d['VitalStartTime'] = min(d['VitalStartTime'], int(elem.attrib['CollectionTimeUTC']))
                    d['VitalEndTime'] = max(d['VitalEndTime'], int(elem.attrib['CollectionTimeUTC']))
            if elem.tag == 'WaveformData':
                attribs = {k:v for k,v in elem.attrib.items() if k in ['Label', 'SampleRate', 'Samples']}
                if d['WaveformData'].get(attribs['Label']):
                    d['WaveformData'][attribs['Label']]['SampleSegments'] = d['WaveformData'][attribs['Label']]['SampleSegments'] + 1
                    d['WaveformData'][attribs['Label']]['SampleRate'] = list(set([int(attribs['SampleRate'])]) | set(d['WaveformData'][attribs['Label']]['SampleRate']))
                    d['WaveformData'][attribs['Label']]['SamplesPerSegment'] = list(set([int(attribs['Samples'])]) | set(d['WaveformData'][attribs['Label']]['SamplesPerSegment']))
                else:
                    d['WaveformData'][attribs['Label']] = {'SampleSegments':1, 'SampleRate':[int(attribs['SampleRate'])], 'SamplesPerSegment':[int(attribs['Samples'])]}
            if elem.tag == 'VitalSign' and elem.find('Parameter') is not None:
                # get vitals data
                attribs = {'DeviceName':[elem.attrib['DeviceName']]}
                for vital in elem:
                    if vital.tag == 'Parameter':
                        attribs = {**attribs, **{'Parameter':vital.text, 'UOM':[]}}
                    if vital.tag == 'Value':
                        attribs = {**attribs, **{'UOM':[vital.attrib['UOM']]}}
                key = attribs['Parameter']
                attribs = {key:{k:v for k,v in attribs.items() if k != 'Parameter'}}
                if key not in d['VitalSign']:
                    attribs[key]['VitalCount'] = 1
                    d['VitalSign'] = {**d['VitalSign'], **attribs}
                else:
                    d['VitalSign'][key]['VitalCount'] = d['VitalSign'][key]['VitalCount'] + 1
                    d['VitalSign'][key]['DeviceName'] = list(set(attribs[key]['DeviceName']) | set(d['VitalSign'][key]['DeviceName']))
                    d['VitalSign'][key]['UOM'] = list(set(attribs[key]['UOM']) | set(d['VitalSign'][key]['UOM']))
            if elem.tag == 'Alarm':
                # count number of alarms
                d['TotalAlarms'] = d['TotalAlarms'] + 1
        else: # event == "end":
            # clean up the tree that is being parsed to save memory
            elem.clear()
            if hasattr(elem, "getprevious"):
                while (
                    elem.getprevious() is not None and elem.getparent() is not None
                ):
                    del elem.getparent()[0]
    d = clean_dict(d) # clean dictionary of lists with single elements
    for time_key in ['WaveformStartTime', 'WaveformEndTime', 'VitalStartTime', 'VitalEndTime']:
        d[time_key] = unix_to_iso(d[time_key])
    return d

def create_bedmaster_header_file(file_path:str, # File path of bedmaster xml file
                                 header_dir:str=None, # Directory to write the file's header file
                                 overwrite:bool=False # Indicator to overwrite the header file, if it already exists in the header_dir
                                 )->None:
    """
    Function to create a file of the bedmaster metadata using the parse_bedmaster_header function. 
    
    Notes
    ----------
    If a header_dir is not passed, the header file will be written in the directory of the file_path.
    """
    file_path = Path(file_path)
    file_path_name = file_path.stem
    if not header_dir:
        file_path_parent = file_path.parent
        header_file_path = file_path_parent/file_path_name
    else:
        header_dir = Path(header_dir)
        header_dir.mkdir(parents=True) if not header_dir.exists() else None
        header_file_path = header_dir/file_path_name
    if not header_file_path.with_suffix('.yaml').exists() or overwrite:
        try:
            parsed_xml_dict = parse_bedmaster_header(file_path)
        except Exception as e:
            print(f"Error parsing file: {file_path}. Error: {e}.", flush=True)
        else:
            try:
                with open((header_file_path).with_suffix('.yaml'), 'w') as f:
                    yaml.dump(parsed_xml_dict, f, default_flow_style=False, sort_keys=False)
            except Exception as e:
                 raise Exception(f"Error dumping file: {file_path}. Error: {e}")

## Dataloaders

In [ ]:
#| export
class ForecastingDataset(Dataset):
    def __init__(self, 
                 channels, # channels to use
                 forecast_window_sec, # forecast window (within), suggest 5, 10, 15 minutes
                 outcome_df, # pandas dataframe containing outcomes for zarr files
                 outcome_df_outcome_col, # outcome column in the y file path
                 file_col='file_path', # column indicating zarr file path
                 y_date_column='date', # column indicating date of sample collection
                 outcome_df_seconds_since_column='Time Stamp (seconds)', # column indicating how many seconds since beginning of waveform
                 outcome_df_duration_column='event_length', # column indicating duration of outcome in seconds
                 sample_df=None, # dataframe indicating which indices within each zarr file includes a sample
                 sample_seq_len_sec=None, # if no sample_df, generate sequences of this length in seconds as one sample
                 frequency=125, # frequency of underlying data
                 butterworth_filters=None, # dictionary of low pass, high pass, and bandpass dictionary to perform on channels
                 median_filter_kernel_size=None, # size of median filter to perform on channels
                 clip_interpolations=None, # dictionary of channels:{'phys_range':..., 'percentiles':...} for filtering and interpolation of filtered values
                 constant_nan_tolerance=0.5, # tolerance for nan values in the data - 0 means no nan allowed, 1 means 100% of nans allowed
                 require_all_channels=False, # indicator to require all channels to be present in the sample, if False, will return samples with any of the channels and 0s for the missing channels
                 infer_forecast_windows=True, # indicator to require all forecast windows to be present in the sample, if False, will return samples with any of the forecast windows and NAs for the missing forecast windows
                 normalize_signals=True, # indicator to normalize signals to 0 mean and unit variance
                 sample_frequency_key='sampling_frequency'
                 ):
        self.channels = channels
        self.forecast_window_sec = [forecast_window_sec] if isinstance(forecast_window_sec, int) else forecast_window_sec
        self.sample_seq_len_sec = sample_seq_len_sec
        self.frequency = frequency
        self.clip_interpolations = clip_interpolations
        self.butterworth_filters = butterworth_filters
        self.median_filter_kernel_size = median_filter_kernel_size
        self.normalize_signals = normalize_signals
        REASON_CODES = {'1': 'Missing channels', 
                        '2': 'No channels', 
                        '3': 'Invalid indices',
                        '4': 'Missing forecast windows', 
                        '5': 'Missing due to nan or constant signal',
                        '6': 'No valid samples'
                        }

       
        if sample_df is None:
            print(f"Calculating samples with {sample_seq_len_sec} sec length using the outcome df with a forecast window of {forecast_window_sec} sec")
            self.sample_df, _, removal_reasons = calculate_samples_forecast_mp(outcome_df=outcome_df, file_col=file_col, outcome_val_col=outcome_df_outcome_col, outcome_time_col=outcome_df_seconds_since_column, outcome_duration_col=outcome_df_duration_column, forecast_window_sec=forecast_window_sec, channels=channels, frequency=frequency, sample_seq_len_sec=sample_seq_len_sec, constant_nan_tolerance=constant_nan_tolerance, require_all_channels=require_all_channels, infer_forecast_windows=infer_forecast_windows, sample_frequency_key=sample_frequency_key)
            self.removal_reasons = {REASON_CODES[k]:v for k,v in removal_reasons.items()}
            print("Samples removed due to the following reasons:")
            for k,v in self.removal_reasons.items():
                print(f"{k}: {v}")
        else:
            self.sample_df = sample_df.copy()
        
        self.outcome_df = outcome_df.copy()
        self.y_date_column = y_date_column
        self.file_col = file_col
        self.forecast_window_sec = forecast_window_sec if isinstance(forecast_window_sec, list) else [forecast_window_sec]
        self.sample_df_outcome_cols = [f'outcome_val_{f}sec' for f in self.forecast_window_sec]

        self.sample_df[self.y_date_column] = self.sample_df[self.file_col].apply(lambda x: dt.datetime.strptime(Path(x).stem.split('-',  maxsplit=1)[1], '%Y-%m-%d-%H-%M'))
        self.sample_df['subject_id'] = self.sample_df[self.file_col].apply(lambda x: Path(x).stem.split('-',  maxsplit=1)[0])
        self.sample_df['unique_identifier'] = self.sample_df['subject_id'].astype(str) + '-' + self.sample_df[self.y_date_column].astype(str)

        # self.outcome_df[self.y_date_column] = pd.to_datetime(self.outcome_df[self.y_date_column])
        # self.outcome_df['unique_identifier'] = self.outcome_df[self.y_mapping_column].astype(str) + '__' + self.outcome_df[self.y_date_column].astype(str)
        # self.outcome_df = self.outcome_df.loc[self.outcome_df['unique_identifier'].isin(self.sample_df['unique_identifier'].unique())]
        # self.outcome_df[y_seconds_since_column] = self.outcome_df[y_seconds_since_column].astype(float)
        # self.outcome_df.set_index(['unique_identifier', self.y_seconds_since_column], inplace=True)
        # self.outcome_df.sort_index(inplace=True)

        # subset_index = pd.MultiIndex.from_arrays([self.sample_df['unique_identifier'], self.sample_df[y_seconds_since_column]])
        # mask = subset_index.isin(self.outcome_df.index)
        # if not mask.all():
        #     warnings.warn(f"There are {sum(~mask)} samples in the zarr files that are not in the outcome df, they will be dropped. Sample unique identifiers: {self.sample_df.loc[~mask, 'unique_identifier'].unique().tolist()}")
        #     self.sample_df = self.sample_df[mask]
        # if self.sample_df.empty:
        #     raise ValueError("There are no samples in the sample_df that match the outcome_df")
        # merge sample and outcome df
        # left = dd.from_pandas(self.sample_df, npartitions=3)
        # right = dd.from_pandas(self.outcome_df, npartitions=3)
        # if not self.forecast_within:
        #     # filter to only exact forecast matches
        #     self.sample_df = dd.merge(left, right, on=['unique_identifier', 'subject_id', 'date', 'Time Stamp (seconds)']).compute()
        # else:
        #     self.sample_df = dd.merge_asof(left, right, on=['unique_identifier', 'subject_id', 'date']).compute()
        self.sample_df.sort_values(by = ['unique_identifier'], ascending=True, inplace=True)
        self.sample_df['start_idx'] = self.sample_df['start_idx'].astype(int)
        self.sample_df['end_idx'] = self.sample_df['end_idx'].astype(int)
        self.total_samples = len(self.sample_df)

    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        # get full length x, idx can be a slice
        sample = self.sample_df.iloc[idx]
        root_grp = zarr.open(sample[self.file_col], mode='r')
        # forecast_max = sample['end_idx'] + self.forecast_window # indice of max window for forecast
        # forecast_max_seconds = forecast_max / self.frequency # number seconds since beginning of waveform
        
        # Get label time (already rounded to nearest minute in __init__)
        Y = sample[self.sample_df_outcome_cols].values.astype(np.int64)
        Y = torch.tensor(Y, dtype=torch.int64)
        signals = []
        for channel in self.channels:
            if channel in root_grp.array_keys():
                temp = root_grp[channel][sample['start_idx']:sample['end_idx']]
                if self.clip_interpolations is not None and channel in self.clip_interpolations:
                    temp = interpolate_nan_clip(temp, physiological_range_clip=self.clip_interpolations[channel]['phys_range'], percentile_clip=self.clip_interpolations[channel]['percentiles'])
                if self.median_filter_kernel_size is not None:
                    temp = median_filter(temp, size=self.median_filter_kernel_size, mode='nearest')
                if self.butterworth_filters is not None and channel in self.butterworth_filters:
                    freq_range = self.butterworth_filters[channel]
                    btype = 'highpass' if freq_range[0] is None else 'lowpass' if freq_range[1] is None else 'bandpass'
                    freq_range = freq_range[1] if freq_range[0] is None else freq_range[0] if freq_range[1] is None else freq_range
                    temp = butterworth(temp, freq_range=freq_range, btype=btype, fs=self.frequency, order=2)
                if self.normalize_signals:
                    temp = iqr_normalization(temp, is_spo2=False)
            else:
                temp = np.zeros((sample['end_idx'] - sample['start_idx'],))
            # if channel == 'ABP':
            #     temp = (temp < 65).astype(np.int32)
            signals.append(temp)
        X = torch.from_numpy(np.array(signals, dtype=np.float32))
        #sequence_padding_mask = torch.zeros([1, X.shape[-1]]) # channels are all the same length
        if torch.isnan(X).any():
            warnings.warn(f"X has nan values, sample_idx: {idx}")
        del root_grp
        return X,Y

In [ ]:
#| export
class BedmasterForecastingDataset(Dataset):
    def __init__(self, 
                 channels, # channels to use
                 forecast_window_sec, # forecast window (within), suggest 5, 10, 15 minutes
                 outcome_df, # pandas dataframe containing outcomes for zarr files
                 outcome_df_outcome_col, # outcome column in the y file path
                 file_col='file_path', # column indicating zarr file path
                 y_date_column='date', # column indicating date of sample collection
                 outcome_df_seconds_since_column='Time Stamp (seconds)', # column indicating how many seconds since beginning of waveform
                 outcome_df_duration_column='event_length', # column indicating duration of outcome in seconds
                 sample_df=None, # dataframe indicating which indices within each zarr file includes a sample
                 sample_seq_len_sec=None, # if no sample_df, generate sequences of this length in seconds as one sample
                 frequency=125, # frequency of underlying data
                 butterworth_filters=None, # dictionary of low pass, high pass, and bandpass dictionary to perform on channels
                 median_filter_kernel_size=None, # size of median filter to perform on channels
                 clip_interpolations=None, # dictionary of channels:{'phys_range':..., 'percentiles':...} for filtering and interpolation of filtered values
                 constant_nan_tolerance=0.5, # tolerance for nan values in the data - 0 means no nan allowed, 1 means 100% of nans allowed
                 require_all_channels=False, # indicator to require all channels to be present in the sample, if False, will return samples with any of the channels and 0s for the missing channels
                 infer_forecast_windows=True, # indicator to require all forecast windows to be present in the sample, if False, will return samples with any of the forecast windows and NAs for the missing forecast windows
                 normalize_signals=True, # indicator to normalize signals to 0 mean and unit variance
                 sample_frequency_key='sample_rate',
                 calibrations=BEDMASTER_CALIBRATIONS
                 ):
        self.channels = channels
        self.forecast_window_sec = [forecast_window_sec] if isinstance(forecast_window_sec, int) else forecast_window_sec
        self.sample_seq_len_sec = sample_seq_len_sec
        self.frequency = frequency
        self.clip_interpolations = clip_interpolations
        self.butterworth_filters = butterworth_filters
        self.median_filter_kernel_size = median_filter_kernel_size
        self.normalize_signals = normalize_signals
        self.sample_frequency_key = sample_frequency_key
        self.calibrations = calibrations
        REASON_CODES = {'1': 'Missing channels', 
                        '2': 'No channels', 
                        '3': 'Invalid indices',
                        '4': 'Missing forecast windows', 
                        '5': 'Missing due to nan or constant signal',
                        '6': 'No valid samples'
                        }

       
        if sample_df is None:
            print(f"Calculating samples with {sample_seq_len_sec} sec length using the outcome df with a forecast window of {forecast_window_sec} sec")
            self.sample_df, _, removal_reasons = calculate_samples_forecast_mp(outcome_df=outcome_df, file_col=file_col, outcome_val_col=outcome_df_outcome_col, outcome_time_col=outcome_df_seconds_since_column, outcome_duration_col=outcome_df_duration_column, forecast_window_sec=forecast_window_sec, channels=channels, frequency=frequency, sample_seq_len_sec=sample_seq_len_sec, constant_nan_tolerance=constant_nan_tolerance, require_all_channels=require_all_channels, infer_forecast_windows=infer_forecast_windows, sample_frequency_key=sample_frequency_key)
            self.removal_reasons = {REASON_CODES[k]:v for k,v in removal_reasons.items()}
            print("Samples removed due to the following reasons:")
            for k,v in self.removal_reasons.items():
                print(f"{k}: {v}")
        else:
            self.sample_df = sample_df.copy()
        
        self.outcome_df = outcome_df.copy()
        self.y_date_column = y_date_column
        self.file_col = file_col
        self.forecast_window_sec = forecast_window_sec if isinstance(forecast_window_sec, list) else [forecast_window_sec]
        self.sample_df_outcome_cols = [f'outcome_val_{f}sec' for f in self.forecast_window_sec]

        # self.outcome_df[self.y_date_column] = pd.to_datetime(self.outcome_df[self.y_date_column])
        # self.outcome_df['unique_identifier'] = self.outcome_df[self.y_mapping_column].astype(str) + '__' + self.outcome_df[self.y_date_column].astype(str)
        # self.outcome_df = self.outcome_df.loc[self.outcome_df['unique_identifier'].isin(self.sample_df['unique_identifier'].unique())]
        # self.outcome_df[y_seconds_since_column] = self.outcome_df[y_seconds_since_column].astype(float)
        # self.outcome_df.set_index(['unique_identifier', self.y_seconds_since_column], inplace=True)
        # self.outcome_df.sort_index(inplace=True)

        # subset_index = pd.MultiIndex.from_arrays([self.sample_df['unique_identifier'], self.sample_df[y_seconds_since_column]])
        # mask = subset_index.isin(self.outcome_df.index)
        # if not mask.all():
        #     warnings.warn(f"There are {sum(~mask)} samples in the zarr files that are not in the outcome df, they will be dropped. Sample unique identifiers: {self.sample_df.loc[~mask, 'unique_identifier'].unique().tolist()}")
        #     self.sample_df = self.sample_df[mask]
        # if self.sample_df.empty:
        #     raise ValueError("There are no samples in the sample_df that match the outcome_df")
        # merge sample and outcome df
        # left = dd.from_pandas(self.sample_df, npartitions=3)
        # right = dd.from_pandas(self.outcome_df, npartitions=3)
        # if not self.forecast_within:
        #     # filter to only exact forecast matches
        #     self.sample_df = dd.merge(left, right, on=['unique_identifier', 'subject_id', 'date', 'Time Stamp (seconds)']).compute()
        # else:
        #     self.sample_df = dd.merge_asof(left, right, on=['unique_identifier', 'subject_id', 'date']).compute()
        self.sample_df['start_idx'] = self.sample_df['start_idx'].astype(int)
        self.sample_df['end_idx'] = self.sample_df['end_idx'].astype(int)
        self.total_samples = len(self.sample_df)

    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        # get full length x, idx can be a slice
        sample = self.sample_df.iloc[idx]
        root_grp = zarr.open(sample[self.file_col],  mode='r')
        # forecast_max = sample['end_idx'] + self.forecast_window # indice of max window for forecast
        # forecast_max_seconds = forecast_max / self.frequency # number seconds since beginning of waveform
        
        # Get label time (already rounded to nearest minute in __init__)
        Y = sample[self.sample_df_outcome_cols].values.astype(np.int64)
        Y = torch.tensor(Y, dtype=torch.int64)
        signals = []
        for channel in self.channels:
            if channel in root_grp.array_keys():
                channel_frequency = int(root_grp[channel].attrs.get(self.sample_frequency_key))
                if channel_frequency != self.frequency:
                    start_idx, end_idx = int(sample['start_idx'] / self.frequency * channel_frequency), int(sample['end_idx'] / self.frequency * channel_frequency)
                else:
                    start_idx, end_idx = sample['start_idx'], sample['end_idx']
                temp = np.array(root_grp[channel][start_idx:end_idx])
                    #temp = resample(root_grp[channel][start_idx:end_idx], self.frequency*self.sample_seq_len_sec) if duration_seconds >= self.sample_seq_len_sec else resample(root_grp[channel][start_idx:end_idx], duration_seconds*self.frequency)
                if channel_frequency != self.frequency:
                    #temp = F_audio.resample(torch.from_numpy(temp), channel_frequency, self.frequency).numpy()
                    temp = resample_waveform(temp, channel_frequency, self.frequency, is_spo2=False)
                if self.calibrations is not None and channel in self.calibrations:
                    temp = temp * self.calibrations[channel]
                if self.clip_interpolations is not None and channel in self.clip_interpolations:
                    temp = interpolate_nan_clip(temp, physiological_range_clip=self.clip_interpolations[channel]['phys_range'], percentile_clip=self.clip_interpolations[channel]['percentiles'])
                if self.median_filter_kernel_size is not None:
                    temp = median_filter(temp, size=self.median_filter_kernel_size, mode='nearest')
                if self.butterworth_filters is not None and channel in self.butterworth_filters:
                    freq_range = self.butterworth_filters[channel]
                    btype = 'highpass' if freq_range[0] is None else 'lowpass' if freq_range[1] is None else 'bandpass'
                    freq_range = freq_range[1] if freq_range[0] is None else freq_range[0] if freq_range[1] is None else freq_range
                    temp = butterworth(temp, freq_range=freq_range, btype=btype, fs=self.frequency, order=2)
                if self.normalize_signals:
                    temp = iqr_normalization(temp, is_spo2=False)
            else:
                temp = np.zeros((sample['end_idx'] - sample['start_idx'],))
            # if channel == 'ABP':
            #     temp = (temp < 65).astype(np.int32)
            signals.append(temp)
        X = torch.from_numpy(np.array(signals, dtype=np.float32))
        #sequence_padding_mask = torch.zeros([1, X.shape[-1]]) # channels are all the same length
        if torch.isnan(X).any():
            warnings.warn(f"X has nan values, sample_idx: {idx}")
        del root_grp
        return X,Y

In [ ]:
#| export
import time
from collections import defaultdict
class BedmasterForecastingDatasetExtended(Dataset):
    def __init__(self, 
                 channels, # channels to use
                 forecast_window_sec, # forecast window (within), suggest 5, 10, 15 minutes
                 outcome_df, # pandas dataframe containing outcomes for zarr files
                 outcome_df_outcome_col, # outcome column in the y file path
                 patch_heartbeats=False, # indicator to patch channels based on heart beats of ECG index channel
                 ecg_index=2, # index of ECG channel in channels list
                 ecg_channel_name='II', # name of ECG channel in zarr file
                 fill_missing_beats=False, # indicator to fill missing heart beats with interpolated values vs leaving as 0s
                 heartbeat_patch_len=256,
                 channel_processing_functions=None, # dictionary of channel processing parameters
                 channel_quality_functions=None, # dictionary of channel quality parameters
                 file_col='file_path', # column indicating zarr file path
                 y_date_column='date', # column indicating date of sample collection
                 outcome_df_seconds_since_column='Time Stamp (seconds)', # column indicating how many seconds since beginning of waveform
                 outcome_df_duration_column='event_length', # column indicating duration of outcome in seconds
                 sample_df=None, # dataframe indicating which indices within each zarr file includes a sample
                 sample_seq_len_sec=None, # if no sample_df, generate sequences of this length in seconds as one sample
                 frequency=125, # frequency of underlying data
                 clip_interpolations=None, # dictionary of channels:{'phys_range':..., 'percentiles':...} for filtering and interpolation of filtered values
                 require_all_channels=False, # indicator to require all channels to be present in the sample, if False, will return samples with any of the channels and 0s for the missing channels
                 infer_forecast_windows=True, # indicator to require all forecast windows to be present in the sample, if False, will return samples with any of the forecast windows and NAs for the missing forecast windows
                 normalize_signals=True, # indicator to normalize signals to 0 mean and unit variance
                 sample_frequency_key='sample_rate',
                 calibrations=BEDMASTER_CALIBRATIONS
                 ):
        self.channels = channels
        self.forecast_window_sec = [forecast_window_sec] if isinstance(forecast_window_sec, int) else forecast_window_sec
        self.sample_seq_len_sec = sample_seq_len_sec
        self.frequency = frequency
        self.clip_interpolations = clip_interpolations
        self.channel_processing_functions = channel_processing_functions
        self.normalize_signals = normalize_signals
        self.sample_frequency_key = sample_frequency_key
        self.calibrations = calibrations
        self.patch_heartbeats = patch_heartbeats
        self.ecg_index = ecg_index
        self.heartbeat_patch_len = heartbeat_patch_len
        self.ecg_channel_name = ecg_channel_name
        self.fill_missing_beats = fill_missing_beats
        REASON_CODES = {'0': 'Valid samples',
                        '1': 'Missing channels', 
                        '2': 'No channels', 
                        '3': 'Invalid indices',
                        '4': 'Missing due to quality functions',
                        }

       
        if sample_df is None:
            print(f"Calculating samples with {sample_seq_len_sec} sec length using the outcome df with a forecast window of {forecast_window_sec} sec")
            self.sample_df, _, removal_reasons = calculate_samples_forecast_df(outcome_df=outcome_df, file_col=file_col, outcome_val_col=outcome_df_outcome_col, outcome_time_col=outcome_df_seconds_since_column, outcome_duration_col=outcome_df_duration_column, forecast_window_sec=forecast_window_sec, channels=channels, frequency=frequency, sample_seq_len_sec=sample_seq_len_sec, channel_quality_functions=channel_quality_functions, require_all_channels=require_all_channels, infer_forecast_windows=infer_forecast_windows, sample_frequency_key=sample_frequency_key)
            self.removal_reasons = {REASON_CODES[k]:v for k,v in removal_reasons.items()}
            print("Samples removed due to the following reasons:")
            for k,v in self.removal_reasons.items():
                print(f"{k}: {v}")
        else:
            self.sample_df = sample_df.copy()
        
        self.outcome_df = outcome_df.copy()
        self.y_date_column = y_date_column
        self.file_col = file_col
        self.forecast_window_sec = forecast_window_sec if isinstance(forecast_window_sec, list) else [forecast_window_sec]
        self.sample_df_outcome_cols = [f'outcome_val_{f}sec' for f in self.forecast_window_sec]

        self.sample_df['start_idx'] = self.sample_df['start_idx'].astype(int)
        self.sample_df['end_idx'] = self.sample_df['end_idx'].astype(int)
        self.total_samples = len(self.sample_df)
        #self.timing_stats = defaultdict(list)

    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        # get full length x, idx can be a slice
        timings = {}
        t_start = time.perf_counter()

        sample = self.sample_df.iloc[idx]

        root_grp = zarr.open(sample[self.file_col],  mode='r')
        # Get label time (already rounded to nearest minute in __init__)
        Y = sample[self.sample_df_outcome_cols].values.astype(np.int64)
        Y = torch.tensor(Y, dtype=torch.int64)
        signals = []
        for channel in self.channels:
            if channel in root_grp.array_keys():
                channel_frequency = int(root_grp[channel].attrs.get(self.sample_frequency_key))
                if channel_frequency != self.frequency:
                    start_idx, end_idx = int(sample['start_idx'] / self.frequency * channel_frequency), int(sample['end_idx'] / self.frequency * channel_frequency)
                else:
                    start_idx, end_idx = sample['start_idx'], sample['end_idx']
                temp = np.array(root_grp[channel][start_idx:end_idx])
                
                t0 = time.perf_counter()
                if channel_frequency != self.frequency:
                    temp = resample_waveform(temp, channel_frequency, self.frequency, is_spo2=False)
                timings[f'{channel}_resample'] = time.perf_counter() - t0
                
                if self.patch_heartbeats and channel == self.ecg_channel_name:
                    t0 = time.perf_counter()
                    ecg_length = len(temp) / self.frequency  # in seconds
                    temp, beat_indices = ecg_clean_beats(temp, fs=self.frequency, filter_ecg=True)
                    n_beats = len(beat_indices) / 2
                    bpm = n_beats / (ecg_length / 60)
                    timings[f'{channel}_ecg_clean_beats'] = time.perf_counter() - t0
                    #temp = resample(root_grp[channel][start_idx:end_idx], self.frequency*self.sample_seq_len_sec) if duration_seconds >= self.sample_seq_len_sec else resample(root_grp[channel][start_idx:end_idx], duration_seconds*self.frequency)
                if self.calibrations is not None and channel in self.calibrations:
                    temp = temp * self.calibrations[channel]
                if self.clip_interpolations is not None and channel in self.clip_interpolations:
                    temp = interpolate_nan_clip(temp, physiological_range_clip=self.clip_interpolations[channel]['phys_range'], percentile_clip=self.clip_interpolations[channel]['percentiles'])
                if self.normalize_signals:
                    temp = iqr_normalization(temp, is_spo2=False)
                if self.channel_processing_functions is not None and channel in self.channel_processing_functions:
                    t0 = time.perf_counter()
                    func = self.channel_processing_functions[channel]
                    temp = func(temp, fs=self.frequency)
                    timings[f'{channel}_custom_func'] = time.perf_counter() - t0
            else:
                temp = np.zeros((sample['end_idx'] - sample['start_idx'],))
            # if channel == 'ABP':
            #     temp = (temp < 65).astype(np.int32)
            signals.append(temp)
        if self.patch_heartbeats:
            # num_patches x nvars x resample len
            t0 = time.perf_counter()
            try:
                signals = create_patch_b2b(np.array(signals, dtype=np.float32), beat_indices=beat_indices, resample_length=self.heartbeat_patch_len, fill_missing_beats=self.fill_missing_beats)
            except Exception as e:
                raise Exception(f"Error creating patches for sample idx {idx}: {e}.")
            timings['create_patch_b2b'] = time.perf_counter() - t0
        else:
            signals = np.array(signals, dtype=np.float32)

        X = torch.from_numpy(signals)
        #sequence_padding_mask = torch.zeros([1, X.shape[-1]]) # channels are all the same length
        if torch.isnan(X).any():
            warnings.warn(f"X has nan values, sample_idx: {idx}")
        del root_grp

        timings['total'] = time.perf_counter() - t_start
    
        # Store timings
        # for k, v in timings.items():
        #     self.timing_stats[k].append(v)

        return X,Y, idx

    def print_timing_stats(self):
        print("\nDataLoader Timing Statistics (averaged over all samples):")
        print("-" * 80)
        
        avg_timings = {k: np.mean(v) * 1000 for k, v in self.timing_stats.items()}  # Convert to ms
        
        for k, v in sorted(avg_timings.items(), key=lambda x: x[1], reverse=True):
            print(f"{k:40s}: {v:8.2f} ms")
        
        print("-" * 80)
        print(f"{'Total samples processed':40s}: {len(next(iter(self.timing_stats.values())))}")

In [ ]:
#| export
class SelfSupervisedDataset(Dataset):
    def __init__(self, 
                 zarr_files, # zarr files that include samples
                 channels, # channels to use
                 max_seq_len_sec=None, # maximum sequence length (in seconds) to use (this is especially relevant when you are returning both stft and raw ts data to keep them in sync)
                 sample_df=None, # dataframe indicating which indices within each zarr file includes a sample
                 sample_seq_len_sec=None, # if no sample_df, generate sequences of this length in seconds as one sample
                 sample_stride_sec=None, #  if no sample_df, seconds of overlap for samples from the same array, if seq_len_seconds == overlap_seconds, there is no overlap
                 frequency=125, # frequency of underlying data
                 butterworth_filters=None, # dictionary of low pass, high pass, and bandpass dictionary to perform on channels
                 median_filter_kernel_size=None, # size of median filter to perform on channels
                 clip_interpolations=None, # dictionary of channels:{'phys_range':..., 'percentiles':...} for filtering and interpolation of filtered values
                 constant_nan_tolerance=0.2, # tolerance for nan values in the data - 0 means no nan allowed, 1 means 100% of nans allowed
                 require_all_channels=True, # indicator to require all channels to be present in the data
                 normalize_signals=True, # indicator to normalize signals to 0 mean and unit variance
                 patch_heartbeats=False, # indicator to patch channels based on heart beats of ECG index channel
                 ecg_index=2, # index of ECG channel in channels list
                 heartbeat_patch_len=256,
                 ):
        self.max_seq_len_sec = max_seq_len_sec
        self.zarr_files = zarr_files
        self.channels = channels
        self.sample_seq_len_sec = sample_seq_len_sec
        self.frequency = frequency
        self.sample_stride_sec = sample_stride_sec
        self.clip_interpolations = clip_interpolations
        self.butterworth_filters = butterworth_filters
        self.median_filter_kernel_size = median_filter_kernel_size
        self.normalize_signals = normalize_signals
        self.constant_nan_tolerance = constant_nan_tolerance
        self.require_all_channels = require_all_channels
        self.patch_heartbeats = patch_heartbeats
        self.ecg_index = ecg_index
        self.heartbeat_patch_len = heartbeat_patch_len
            
        if sample_df is None:
            assert sample_seq_len_sec is not None and sample_stride_sec is not None, "You must provide sample sequence lengths and strides if you do not pass a sample_df"
            print(f"Calculating samples with {sample_seq_len_sec} sec length and {sample_stride_sec} sec stride")
            self.sample_df, _ = calculate_samples_mp(zarr_files, channels=channels, max_seq_len_sec=max_seq_len_sec, sample_seq_len_sec=sample_seq_len_sec, frequency=frequency, stride_sec=sample_stride_sec, include_partial_samples=False, constant_nan_tolerance=constant_nan_tolerance, require_all_channels=require_all_channels)
        else:
            self.sample_df = sample_df.copy()

        self.sample_df['start_idx'] = self.sample_df['start_idx'].astype(int)
        self.sample_df['end_idx'] = self.sample_df['end_idx'].astype(int)
        self.total_samples = len(self.sample_df)

    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        # get full length x, idx can be a slice
        sample = self.sample_df.iloc[idx]
        root_grp = zarr.open(sample['file'], mode='r')
        
        signals = []
        for channel in self.channels:
            temp = root_grp[channel][sample['start_idx']:sample['end_idx']]
            if self.clip_interpolations is not None and channel in self.clip_interpolations:
                temp = interpolate_nan_clip(temp, physiological_range_clip=self.clip_interpolations[channel]['phys_range'], percentile_clip=self.clip_interpolations[channel]['percentiles'])
            if self.median_filter_kernel_size is not None:
                temp = median_filter(temp, size=self.median_filter_kernel_size, mode='nearest')
            if self.butterworth_filters is not None and channel in self.butterworth_filters:
                freq_range = self.butterworth_filters[channel]
                btype = 'highpass' if freq_range[0] is None else 'lowpass' if freq_range[1] is None else 'bandpass'
                freq_range = freq_range[1] if freq_range[0] is None else freq_range[0] if freq_range[1] is None else freq_range
                temp = butterworth(temp, freq_range=freq_range, btype=btype, fs=self.frequency, order=2)
            if self.normalize_signals:
                temp = iqr_normalization(temp, is_spo2=False)
            signals.append(temp)
        if self.patch_heartbeats:
            # num_patches x nvars x resample len
            signals = create_patch_b2b(np.array(signals, dtype=np.float32), ecg_index=self.ecg_index, frequency=self.frequency, resample_length=self.heartbeat_patch_len, fill_missing_beats=True)
        else:
            signals = np.array(signals, dtype=np.float32)
        
        X = Y = torch.from_numpy(signals)
        if torch.isnan(X).any():
            warnings.warn(f"X has nan values, sample_idx: {idx}")
        return X,Y

In [ ]:
#| export
def nested_patches_ss_tensor_collate(batch):
    """
    Collate function for variable length sequences using NestedTensor.
    
    Args:
        batch: List of tuples (X, Y) where:
            X: Tensor of shape (num_patches, channels, patch_len)
            Y: Tensor of shape (num_patches, channels, patch_len)
            
    Returns:
        X_nested: NestedTensor containing the batch of sequences
        Y_nested: NestedTensor containing the batch of sequences
    """
    # Separate X and Y from batch
    X, Y = zip(*batch)
    
    # Convert to nested tensor
    ## the transpose is to make the jagged dimension the first dimension (which is the sequence length)
    ## if batches all have the same sequence length, torch automatically makes the jagged dimension the first dimension

    X_nested = torch.nested.as_nested_tensor(X, layout=torch.jagged)
    Y_nested = torch.nested.as_nested_tensor(Y, layout=torch.jagged)
    return X_nested, Y_nested

In [ ]:
#| export
def nested_patches_y_tensor_collate(batch):
    """
    Collate function for variable length sequences using NestedTensor.
    
    Args:
        batch: List of tuples (X, Y) where:
            X: Tensor of shape (channels, seq_len)
            Y: single outcome (bs)
            
    Returns:
        X_nested: NestedTensor containing the batch of sequences
        Y: Tensor containing the batch of targets
    """
    # Separate X and Y from batch
    X, Y, *idx = zip(*batch)
    
    # Convert to nested tensor
    ## the transpose is to make the jagged dimension the first dimension (which is the sequence length)
    ## if batches all have the same sequence length, torch automatically makes the jagged dimension the first dimension

    X_nested = torch.nested.as_nested_tensor(X, layout=torch.jagged)
    Y = torch.stack(Y, dim=0)
    if len(idx) > 0:
        idx = idx[0]
        idx = torch.tensor(idx, dtype=torch.int64).squeeze(-1)
        return X_nested, Y, idx
    else:
        return X_nested, Y

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()